# LAB 1: Índice Híbrido no OpenSearch com Ollama BGE-M3 (Connector ML Commons)
## Busca Híbrida kNN + BM25 para Corpus Jurídico (TCU 2026, 1.000+ acórdãos)

**Objetivo:** Criar um índice híbrido no OpenSearch 3.x que combina **busca vetorial (kNN, BGE-M3 dim=1024)** com **busca lexical (BM25)** para o corpus jurídico enriquecido `corpus_juridico_aula4_v2.json` (1.100 acórdãos do TCU).

**Abordagem:** Conectamos o OpenSearch ao **Ollama** via **Connector do ML Commons Plugin**, registramos o modelo `bge-m3` e usamos um **ingest pipeline** para gerar embeddings **server-side** durante a ingestão — sem precisar enviar vetores do cliente.

**Referência:**
- LOMBARDO, A. *Integrating Ollama and OpenSearch for Vector Indexing Using Models from Ollama*. LinkedIn, 2024. <https://www.linkedin.com/pulse/integrating-ollama-opensearch-vector-indexing-using-models-lombardo-anmie/>
- OpenSearch Docs: *Connecting to externally hosted models*. <https://docs.opensearch.org/3.0/ml-commons-plugin/remote-models/index/>

**Pré-requisitos:**
- OpenSearch 3.x + ML Commons + Neural Search plugins
- Ollama rodando em `http://host.docker.internal:11434` (ou IP da rede) com `ollama pull bge-m3`
- Corpus em `aula4/datasets/corpus_juridico_aula4_v2.json` (1.100 acórdãos TCU 2026)

**Duração estimada:** 60 minutos

## 1. Instalação de Dependências

In [1]:
import subprocess, sys
for pkg in ['opensearch-py>=2.7', 'requests', 'tqdm', 'python-dotenv', 'pandas']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Dependências instaladas.')

Dependências instaladas.


## 2. Configuração do Ambiente

In [2]:
import json, os, warnings, requests
from opensearchpy import OpenSearch
from tqdm import tqdm
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
load_dotenv()

OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST', 'localhost')
OPENSEARCH_PORT = int(os.getenv('OPENSEARCH_PORT', 9200))
OPENSEARCH_USER = os.getenv('OPENSEARCH_USER', 'admin')
OPENSEARCH_PASS = os.getenv('OPENSEARCH_PASS', 'admin')
USE_SSL         = os.getenv('OPENSEARCH_USE_SSL', 'False').lower() == 'true'

# Endpoint Ollama VISTO PELO container OpenSearch.
# - Docker Desktop: host.docker.internal:11434
# - Linux/host net: 172.17.0.1:11434
EMBED_MODEL = 'bge-m3'
EMBED_DIM   = 1024  # BGE-M3

client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=USE_SSL, verify_certs=False, ssl_show_warn=False, http_compress=True,
)

info = client.info()
print(f"OpenSearch {info['version']['number']} OK")
print(f"Modelo de embeddings: {EMBED_MODEL} (dim={EMBED_DIM})")

OpenSearch 3.0.0 OK
Modelo de embeddings: bge-m3 (dim=1024)


## 3. Configurar ML Commons (Permitir Conectores Externos)

Necessário antes de registrar conectores apontando para hosts externos (Ollama).

In [3]:
import re
import json

settings_body = {
    'persistent': {
        'plugins.ml_commons.only_run_on_ml_node': 'false',
        'plugins.ml_commons.model_access_control_enabled': 'true',
        'plugins.ml_commons.native_memory_threshold': '99',
        'plugins.ml_commons.allow_registering_model_via_url': 'true',
        'plugins.ml_commons.connector.private_ip_enabled': 'true',
        'plugins.ml_commons.trusted_connector_endpoints_regex': [
            '^https?://host\\.docker\\.internal:11434/.*$',
            '^https?://localhost:11434/.*$',
            '^https?://10\\.93\\.[0-9]{1,3}\\.[0-9]{1,3}(:[0-9]+)?/.*$',
            '^https?://192\\.168\\.1\\.200:11434/.*$',
        ],
    },
    'transient': {}
}

# Envia a requisição para o cluster
resp = client.transport.perform_request('PUT', '/_cluster/settings', body=settings_body)

print('Cluster settings atualizadas:')
print(json.dumps(resp, indent=2)[:400])

Cluster settings atualizadas:
{
  "acknowledged": true,
  "persistent": {
    "plugins": {
      "ml_commons": {
        "only_run_on_ml_node": "false",
        "trusted_connector_endpoints_regex": [
          "^https?://host\\.docker\\.internal:11434/.*$",
          "^https?://localhost:11434/.*$",
          "^https?://10\\.93\\.[0-9]{1,3}\\.[0-9]{1,3}(:[0-9]+)?/.*$",
          "^https?://192\\.168\\.1\\.200:11434/.*$"
      


## 4. Registrar Connector Ollama no ML Commons

O connector descreve como chamar o endpoint `POST /api/embeddings` do Ollama e como mapear a resposta JSON.

In [4]:
connector_body = {
    'name': 'ollama-connector-novo',
    'description': 'Connector corrigido usando a API OpenAI do Ollama',
    'version': 1,
    'protocol': 'http',
    'parameters': {
        'endpoint': 'http://192.168.1.200:11434',
        'model': 'bge-m3'
    },
    'credential': {
        'openAI_key': 'dummy'
    },
    'actions': [
        {
            'action_type': 'predict',
            'method': 'POST',
            'url': '${parameters.endpoint}/v1/embeddings',
            'headers': {'Content-Type': 'application/json'},
            'request_body': '{ "model": "${parameters.model}", "input": ${parameters.input} }', 
            'pre_process_function': 'connector.pre_process.openai.embedding',
            'post_process_function': 'connector.post_process.openai.embedding'
        }
    ]
}

resp = client.transport.perform_request(
    'POST', '/_plugins/_ml/connectors/_create', body=connector_body)
connector_id = resp['connector_id']
print(f'Connector criado: {connector_id}')

Connector criado: jLkcip4B7ffnRP14tX_e


## 5. Registrar e Deployar o Modelo no ML Commons

In [5]:
import time

model_body = {
    'name': 'ollama-bge-m3',
    'function_name': 'remote',
    'description': 'BGE-M3 servido pelo Ollama via connector remoto',
    'connector_id': connector_id,
}
resp = client.transport.perform_request(
    'POST', '/_plugins/_ml/models/_register?deploy=true', body=model_body)
task_id = resp['task_id']
print(f'Tarefa de registro: {task_id}')

# Polling do status
model_id = None
for _ in range(40):
    s = client.transport.perform_request('GET', f'/_plugins/_ml/tasks/{task_id}')
    if s.get('state') == 'COMPLETED':
        model_id = s.get('model_id')
        break
    if s.get('state') == 'FAILED':
        raise RuntimeError(f"Registro falhou: {s}")
    time.sleep(2)

print(f'Modelo registrado e deployado: model_id = {model_id}')

Tarefa de registro: jbkdip4B7ffnRP14N380
Modelo registrado e deployado: model_id = jrkdip4B7ffnRP14N39p


## 6. Smoke-test do Modelo Remoto

In [7]:
predict_body = {
  "parameters": {
    "input": ["Seu texto de teste aqui"]
  }
}

resp = client.transport.perform_request(
    'POST', f'/_plugins/_ml/models/{model_id}/_predict', body=predict_body)
# Estrutura: inference_results[0].output[0].data → vetor
out = resp['inference_results'][0]['output'][0]['data']
print(out) 
print(f'Dim do embedding retornado: {len(out)} (esperado {EMBED_DIM})')
assert len(out) == EMBED_DIM, 'Dimensão inesperada!'

[-0.05725016, 0.018520396, -0.008649232, -0.033622723, -0.0017255951, -0.057409577, 0.0073398286, 0.039733477, 0.0070243836, 0.020955842, 0.011723425, -0.003646483, -0.016453594, 0.0044974177, 0.0009070685, -0.007910795, -0.008743194, -0.009917316, -0.048547998, -0.022505485, -0.029317714, -0.0033800174, 0.052913718, 0.032974657, -0.010704723, 0.030527763, 0.0009164995, 0.010518554, -0.0062349625, 0.03799639, 0.034242384, -0.015216825, 0.0006292736, -0.05524987, -0.0154606635, -0.02688148, 0.0061220075, -0.042055085, -0.02824263, 0.017204478, 0.06555481, -0.035410106, -0.0040226276, -0.014207776, 0.01940915, -0.008646341, 0.030731117, -0.007004937, -0.023827944, -0.0016867641, -0.014138183, 0.021317268, 0.043599654, -0.029144803, -0.014030298, 0.013319817, 0.023081904, -0.0042978455, -0.046767186, 0.032047335, -0.009204479, -0.004126016, 0.021036701, 0.027949328, 0.013208144, 0.018458387, 0.031552266, -0.025946721, -0.010554339, -0.006142145, 0.0031118558, 0.02450556, -0.052903622, -0.

## 7. Criar Ingest Pipeline Neural

O **ingest pipeline** intercepta a indexação e gera embeddings do campo `conteudo` automaticamente, populando `knn_vector`.

In [8]:
# Aqui o opensearch faz a chamada no Ollama para embeding dos Acordãos 
INGEST_PIPELINE = 'pipeline_bge_m3_ingest'
ingest_body = {
    'description': 'Gera embeddings BGE-M3 (Ollama) durante a ingestão',
    'processors': [
        {
            'text_embedding': {
                'model_id': model_id,
                'field_map': {
                    'conteudo': 'knn_vector'
                }
            }
        }
    ]
}
client.transport.perform_request('PUT', f'/_ingest/pipeline/{INGEST_PIPELINE}', body=ingest_body)
print(f'Ingest pipeline criado: {INGEST_PIPELINE}')

Ingest pipeline criado: pipeline_bge_m3_ingest


## 8. Criar o Índice Híbrido (kNN + BM25, dim=1024)

In [9]:
INDEX_NAME = 'corpus_juridico_aula4'

index_body = {
    'settings': {
        'index': {
            'knn': True,
            'knn.algo_param.ef_search': 100,
            'default_pipeline': INGEST_PIPELINE,
            'number_of_shards': 1,
            'number_of_replicas': 0,
        },
        'analysis': {
            'analyzer': {
                'portuguese_analyzer': {'type': 'standard', 'stopwords': '_portuguese_'}
            }
        }
    },
    'mappings': {
        'properties': {
            'id':        {'type': 'keyword'},
            'tipo':      {'type': 'keyword'},
            'titulo':    {'type': 'text', 'analyzer': 'portuguese_analyzer'},
            'conteudo':  {'type': 'text', 'analyzer': 'portuguese_analyzer'},
            'metadata':  {'type': 'object', 'enabled': True},
            'knn_vector': {
                'type': 'knn_vector',
                'dimension': EMBED_DIM,
                'method': {
                    'name': 'hnsw',
                    'space_type': 'cosinesimil',
                    'engine': 'faiss',
                    'parameters': {'ef_construction': 128, 'm': 16}
                }
            }
        }
    }
}

if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)
    print('Índice anterior removido.')
client.indices.create(index=INDEX_NAME, body=index_body)
print(f'Índice {INDEX_NAME} criado (dim={EMBED_DIM}, pipeline={INGEST_PIPELINE}).')

Índice anterior removido.
Índice corpus_juridico_aula4 criado (dim=1024, pipeline=pipeline_bge_m3_ingest).


## 9. Carregar o Corpus Jurídico Enriquecido (TCU 2026)

Usamos `corpus_juridico_aula4_v2.json` (≥1.000 acórdãos extraídos de `aula2/datasets/acordao-completo-2026.csv`). Se ausente, usa o corpus pequeno legado como fallback.

In [10]:
CORPUS_PRIM = '../datasets/corpus_juridico_aula4_v2.json'
CORPUS_LEG  = '../datasets/corpus_juridico_aula4.json'

corpus_path = CORPUS_PRIM if os.path.exists(CORPUS_PRIM) else CORPUS_LEG
with open(corpus_path, 'r', encoding='utf-8') as f:
    corpus = json.load(f)
print(f'Corpus carregado: {corpus_path}')
print(f'Total de documentos: {len(corpus)}')
print(f'Tipos únicos: {len(set(d.get("tipo","?") for d in corpus))}')
print(f'Exemplo:\n  id: {corpus[0]["id"]}\n  titulo: {corpus[0]["titulo"][:80]}...')

Corpus carregado: ../datasets/corpus_juridico_aula4_v2.json
Total de documentos: 1100
Tipos únicos: 2
Exemplo:
  id: acordao_2026_2344_2764954
  titulo: ACÓRDÃO DE RELAÇÃO 2344/2026 ATA 15/2026 - SEGUNDA CÂMARA...


## 10. Ingestão em Bulk (embeddings gerados server-side via pipeline)

Cada documento é submetido **sem** o campo `knn_vector` — o ingest pipeline chama o Ollama via connector e popula o vetor automaticamente.

In [11]:
from tqdm import tqdm

BATCH = 25  # menor para não saturar o Ollama em CPU
total = len(corpus)
erros = 0

for start in tqdm(range(0, total, BATCH), desc='Bulk ingest'):
    sub = corpus[start:start + BATCH]
    bulk = []
    for doc in sub:
        bulk.append({'index': {'_index': INDEX_NAME, '_id': doc['id']}})
        # NÃO enviar knn_vector — pipeline gera no servidor
        bulk.append({
            'id':       doc['id'],
            'tipo':     doc.get('tipo', ''),
            'titulo':   doc.get('titulo', ''),
            'conteudo': doc.get('conteudo', ''),
            'metadata': doc.get('metadata', {}),
        })
    
    try:
        resp = client.bulk(body=bulk, request_timeout=120)
        
        # Verifica se houve erros no lote
        if resp.get('errors'):
            for item in resp['items']:
                op_status = item.get('index', {})
                error_details = op_status.get('error')
                
                if error_details:
                    erros += 1
                    doc_id = op_status.get('_id')
                    print(f"\n[ERRO] Falha no documento ID: {doc_id}")
                    print(f"Tipo do erro: {error_details.get('type')}")
                    print(f"Razão: {error_details.get('reason')}")
                    # Caso o erro seja causado por outro componente (ex: falha do Ollama no pipeline)
                    if error_details.get('caused_by'):
                        print(f"Causado por: {error_details['caused_by'].get('reason')}")
                    print("-" * 50)
                    
    except Exception as e:
        print(f"\n[ERRO CRÍTICO] Falha ao enviar o lote ou timeout: {e}")
        erros += len(sub) # Assume que o lote inteiro falhou se a requisição estourar

client.indices.refresh(index=INDEX_NAME)
count = client.count(index=INDEX_NAME)['count']
print(f'\nDocumentos no índice: {count}/{total} | erros detectados: {erros}')

Bulk ingest: 100%|██████████| 44/44 [01:54<00:00,  2.60s/it]



Documentos no índice: 1100/1100 | erros detectados: 0


## 11. Validação — kNN e BM25

In [ ]:
QUERY = 'operação de crédito com garantia da União'

# Para kNN no cliente, podemos usar neural query (server-side) que evita gerar embedding aqui:
neural_query = {
    'size': 5,
    'query': {
        'neural': {
            'knn_vector': {
                'query_text': QUERY,
                'model_id':   model_id,
                'k': 5,
            }
        }
    },
    '_source': ['id', 'titulo', 'tipo']
}
resp = client.search(index=INDEX_NAME, body=neural_query)
print(f'\nNeural kNN — "{QUERY}":')
for h in resp['hits']['hits']:
    print(f"  [{h['_score']:.4f}] {h['_source']['titulo'][:90]}")

# BM25
bm25 = {
    'size': 5,
    'query': {'match': {'conteudo': {'query': QUERY}}},
    '_source': ['id', 'titulo', 'tipo']
}
resp = client.search(index=INDEX_NAME, body=bm25)
print(f'\nBM25 — "{QUERY}":')
for h in resp['hits']['hits']:
    print(f"  [{h['_score']:.4f}] {h['_source']['titulo'][:90]}")

## 12. Persistir IDs para os Próximos LABs

Os LABs 2/3/5/6 dependem de `INDEX_NAME` e `model_id`. Salvamos em arquivo.

In [ ]:
config_out = {
    'index_name':   INDEX_NAME,
    'model_id':     model_id,
    'connector_id': connector_id,
    'embed_dim':    EMBED_DIM,
    'embed_model':  EMBED_MODEL,
    'ingest_pipeline': INGEST_PIPELINE,
}
with open('lab1_config.json', 'w', encoding='utf-8') as f:
    json.dump(config_out, f, indent=2, ensure_ascii=False)
print('lab1_config.json gravado:')
print(json.dumps(config_out, indent=2))

## Exercício Proposto

1. Modifique o `field_map` do ingest pipeline para gerar embeddings também do campo `titulo` em um novo campo `titulo_vector` (dim=1024).
2. Reindexe alguns documentos e compare a relevância de uma busca kNN no campo `conteudo` vs. `titulo`.
3. Discuta o trade-off: indexar título separadamente melhora MRR em buscas curtas ("habeas corpus")?

## Referências (ABNT)

LOMBARDO, A. **Integrating Ollama and OpenSearch for Vector Indexing Using Models from Ollama**. LinkedIn Pulse, 2024. Disponível em: <https://www.linkedin.com/pulse/integrating-ollama-opensearch-vector-indexing-using-models-lombardo-anmie/>.

OPENSEARCH PROJECT. **Connecting to externally hosted models**. OpenSearch 3.0 Docs. Disponível em: <https://docs.opensearch.org/3.0/ml-commons-plugin/remote-models/index/>.

OPENSEARCH PROJECT. **Neural search**. OpenSearch 3.0 Docs. Disponível em: <https://docs.opensearch.org/3.0/vector-search/ai-search/neural-search/>.

CHEN, J. et al. **BGE M3-Embedding: Multi-Lingual, Multi-Functionality, Multi-Granularity Text Embeddings**. arXiv:2309.07597, 2024.

TRIBUNAL DE CONTAS DA UNIÃO (TCU). **Acórdãos 2026 — base completa**. Brasília: TCU, 2026.